# **Laboratorio 6. Análisis de redes sociales**

## **Ejercicio 2. Calidad, limpieza y preprocesamiento**

---

## **2.1. Diagnóstico inicial de calidad**

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW = RAIZ / "data" / "raw"

OPCIONES_LECTURA = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")

videos = pd.read_csv(DIR_RAW / "youtube_videos.csv", **OPCIONES_LECTURA)
comentarios = pd.read_csv(DIR_RAW / "youtube_comments.csv", **OPCIONES_LECTURA)

print("DIMENSIONES")
print(f"  youtube_videos.csv    : {videos.shape[0]} filas x {videos.shape[1]} columnas")
print(f"  youtube_comments.csv  : {comentarios.shape[0]} filas x {comentarios.shape[1]} columnas")
print(f"  celdas totales        : {videos.size + comentarios.size}")

DIMENSIONES
  youtube_videos.csv    : 293 filas x 20 columnas
  youtube_comments.csv  : 406 filas x 17 columnas
  celdas totales        : 12762


In [2]:
TIPOS_VIDEOS = {
    "video_id": ("cualitativa", "nominal"), "title": ("cualitativa", "nominal"),
    "channel_name": ("cualitativa", "nominal"), "channel_id": ("cualitativa", "nominal"),
    "source_query": ("cualitativa", "nominal"), "source_group": ("cualitativa", "nominal"),
    "dataset_sources": ("cualitativa", "nominal"), "channel_handle": ("cualitativa", "nominal"),
    "published_time": ("cualitativa", "ordinal"), "view_count_text": ("cuantitativa", "discreta"),
    "description_snippet": ("cualitativa", "nominal"), "video_url": ("cualitativa", "nominal"),
    "query_hits": ("cualitativa", "nominal"), "keywords": ("cualitativa", "nominal"),
    "description": ("cualitativa", "nominal"), "view_count": ("cuantitativa", "discreta"),
    "publish_date": ("cuantitativa", "continua"), "upload_date": ("cuantitativa", "continua"),
    "category": ("cualitativa", "nominal"), "owner_handle": ("cualitativa", "nominal"),
}

TIPOS_COMENTARIOS = {
    "video_id": ("cualitativa", "nominal"), "comment_id": ("cualitativa", "nominal"),
    "video_title": ("cualitativa", "nominal"), "channel_name": ("cualitativa", "nominal"),
    "channel_id": ("cualitativa", "nominal"), "author_name": ("cualitativa", "nominal"),
    "author_channel_id": ("cualitativa", "nominal"), "text": ("cualitativa", "nominal"),
    "source_query": ("cualitativa", "nominal"), "source_group": ("cualitativa", "nominal"),
    "dataset_sources": ("cualitativa", "nominal"), "author_handle": ("cualitativa", "nominal"),
    "published_text": ("cualitativa", "ordinal"), "like_count_text": ("cuantitativa", "discreta"),
    "reply_count": ("cuantitativa", "discreta"), "is_pinned": ("cualitativa", "nominal"),
    "viewer_rating": ("cuantitativa", "discreta"),
}


def tabla_tipos(df, tipos):
    return pd.DataFrame({
        "columna": df.columns,
        "tipo": [tipos[c][0] for c in df.columns],
        "subtipo": [tipos[c][1] for c in df.columns],
        "tipo_pandas": [str(df[c].dtype) for c in df.columns],
        "distintos": [df[c].nunique() for c in df.columns],
        "ejemplo": [df[c].iloc[0][:38] for c in df.columns],
    })


tabla_tipos(videos, TIPOS_VIDEOS)

,columna,tipo,subtipo,tipo_pandas,distintos,ejemplo
0,video_id,cualitativa,nominal,str,293,-5puKGEqcUc
1,title,cualitativa,nominal,str,274,INSIVUMEH pronostica incremento de llu
2,channel_name,cualitativa,nominal,str,97,T13 Noticias Guatemala
3,channel_id,cualitativa,nominal,str,97,UCq0Cm-3SKthEySQc2JZBi1A
4,source_query,cualitativa,nominal,str,21,guatemala lluvias
5,source_group,cualitativa,nominal,str,3,topic
6,dataset_sources,cualitativa,nominal,str,23,youtube_guatemala.csv | youtube_guatem
7,channel_handle,cualitativa,nominal,str,97,/@T13NoticiasGuatemala
8,published_time,cualitativa,ordinal,str,81,hace 2 días
9,view_count_text,cuantitativa,discreta,str,260,"2,390 vistas"


In [3]:
tabla_tipos(comentarios, TIPOS_COMENTARIOS)

,columna,tipo,subtipo,tipo_pandas,distintos,ejemplo
0,video_id,cualitativa,nominal,str,19,j43HgwYFKfk
1,comment_id,cualitativa,nominal,str,406,Ugw-J65a1iYL9hqhELh4AaABAg
2,video_title,cualitativa,nominal,str,19,La cooptación de Walter Mazariegos en
3,channel_name,cualitativa,nominal,str,8,Quorum
4,channel_id,cualitativa,nominal,str,8,UCE4rsXcgDb6e1-a9iTbWzfg
5,author_name,cualitativa,nominal,str,332,@MarcosCarillo-b1r
6,author_channel_id,cualitativa,nominal,str,332,UCdFlugHJJa4l3YqWuNRmvXw
7,text,cualitativa,nominal,str,404,Ese corrupto amigo de la vieja fiscal
8,source_query,cualitativa,nominal,str,6,@quorumgt
9,source_group,cualitativa,nominal,str,2,topic


In [4]:
todos = list(TIPOS_VIDEOS.values()) + list(TIPOS_COMENTARIOS.values())

print("RESUMEN DE TIPOS (37 variables entre los dos archivos)")
print(f"  cualitativas nominales  : {sum(1 for t, s in todos if s == 'nominal')}")
print(f"  cualitativas ordinales  : {sum(1 for t, s in todos if s == 'ordinal')}")
print(f"  cuantitativas discretas : {sum(1 for t, s in todos if s == 'discreta')}")
print(f"  cuantitativas continuas : {sum(1 for t, s in todos if s == 'continua')}")
print()
print(f"  tipos de pandas presentes: {set(str(videos[c].dtype) for c in videos.columns) | set(str(comentarios[c].dtype) for c in comentarios.columns)}")

RESUMEN DE TIPOS (37 variables entre los dos archivos)
  cualitativas nominales  : 28
  cualitativas ordinales  : 2
  cuantitativas discretas : 5
  cuantitativas continuas : 2

  tipos de pandas presentes: {'str'}


In [5]:
def faltantes_por_marcador(df, nombre_df):
    """Los faltantes no vienen como NaN sino como cadena vacia, espacios o lista vacia."""
    filas = []
    for col in df.columns:
        s = df[col]
        cadena_vacia = int((s == "").sum())
        solo_espacios = int(((s != "") & (s.str.strip() == "")).sum())
        lista_vacia = int((s.str.strip() == "[]").sum())
        total = cadena_vacia + solo_espacios + lista_vacia
        if total:
            filas.append({
                "columna": col, "cadena_vacia": cadena_vacia, "solo_espacios": solo_espacios,
                "lista_vacia": lista_vacia, "total": total,
                "pct": round(100 * total / len(df), 1),
            })
    print(f"--- {nombre_df} ---")
    return pd.DataFrame(filas).sort_values("total", ascending=False).reset_index(drop=True)


faltantes_por_marcador(videos, "youtube_videos.csv")

--- youtube_videos.csv ---


,columna,cadena_vacia,solo_espacios,lista_vacia,total,pct
0,keywords,0,0,162,162,55.3
1,description,26,0,0,26,8.9
2,description_snippet,25,0,0,25,8.5
3,published_time,13,0,0,13,4.4
4,view_count_text,13,0,0,13,4.4


In [6]:
faltantes_por_marcador(comentarios, "youtube_comments.csv")

--- youtube_comments.csv ---


,columna,cadena_vacia,solo_espacios,lista_vacia,total,pct
0,viewer_rating,406,0,0,406,100.0
1,like_count_text,0,189,0,189,46.6


In [7]:
print("DUPLICADOS EN videos")
print(f"  filas completas duplicadas      : {int(videos.duplicated().sum())}")
print(f"  video_id duplicado              : {int(videos.video_id.duplicated().sum())}")
print(f"  title duplicado                 : {int(videos.title.duplicated().sum())}")
print(f"  (title, channel_id) duplicado   : {int(videos.duplicated(subset=['title', 'channel_id']).sum())}")
print()

print("DUPLICADOS EN comentarios")
print(f"  filas completas duplicadas      : {int(comentarios.duplicated().sum())}")
print(f"  comment_id duplicado            : {int(comentarios.comment_id.duplicated().sum())}")
print(f"  text duplicado                  : {int(comentarios.text.duplicated().sum())}")
print(f"  (text, author_channel_id) dup   : {int(comentarios.duplicated(subset=['text', 'author_channel_id']).sum())}")

DUPLICADOS EN videos
  filas completas duplicadas      : 0
  video_id duplicado              : 0
  title duplicado                 : 19
  (title, channel_id) duplicado   : 19

DUPLICADOS EN comentarios
  filas completas duplicadas      : 0
  comment_id duplicado            : 0
  text duplicado                  : 2
  (text, author_channel_id) dup   : 2


In [8]:
titulos_repetidos = (
    videos[videos.duplicated(subset=["title", "channel_id"], keep=False)]
    .groupby(["channel_name", "title"])
    .agg(n_videos=("video_id", "size"))
    .sort_values("n_videos", ascending=False)
    .reset_index()
)
titulos_repetidos

,channel_name,title,n_videos
0,Diario de Centro América,#EnVivoDCA | Conferencia de prensa del Gobiern...,6
1,Gobierno de la República de Guatemala,Conferencia de Prensa del Gobierno de Guatemal...,6
2,Gobierno de la República de Guatemala,Conferencia de Prensa del Gobierno de Guatemal...,5
3,Diario de Centro América,#EnVivoDCA | Conferencia de Prensa del Gobiern...,2
4,Gobierno de la República de Guatemala,Conferencia de Prensa del Gobierno de Guatemal...,2
5,Gobierno de la República de Guatemala,Noticiero El Informativo. Nuestra semana de re...,2
6,Municipalidad de Guatemala,"Ciudad de Guatemala, el mejor lugar para vivir",2
7,PNCdeGuatemala,Acciones relevantes en el #Resumen24Horas,2


In [9]:
print("VARIABLES CONSTANTES")
for nombre, df in [("videos", videos), ("comentarios", comentarios)]:
    constantes = [c for c in df.columns if df[c].nunique() <= 1]
    print(f"  {nombre}: {constantes if constantes else 'ninguna'}")
    for c in constantes:
        print(f"      {c} -> valor unico: {df[c].unique().tolist()}")

VARIABLES CONSTANTES
  videos: ninguna
  comentarios: ['is_pinned', 'viewer_rating']
      is_pinned -> valor unico: ['False']
      viewer_rating -> valor unico: ['']


In [10]:
def atipicos_iqr(serie, nombre):
    """Regla de Tukey: atipico si cae fuera de 1.5 veces el rango intercuartilico."""
    s = pd.to_numeric(serie, errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    ric = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * ric, q3 + 1.5 * ric
    fuera = s[(s < limite_inf) | (s > limite_sup)]
    print(f"{nombre}")
    print(f"    n={len(s)}  min={s.min():.0f}  mediana={s.median():.0f}  media={s.mean():.1f}  max={s.max():.0f}")
    print(f"    Q1={q1:.0f}  Q3={q3:.0f}  IQR={ric:.0f}  limite_sup={limite_sup:.0f}")
    print(f"    atipicos: {len(fuera)}  ({len(fuera) / len(s):.1%})")
    print(f"    cinco mayores: {s.nlargest(5).tolist()}")
    print()


atipicos_iqr(videos.view_count, "videos.view_count")
atipicos_iqr(comentarios.reply_count, "comentarios.reply_count")
atipicos_iqr(comentarios.like_count_text.replace(r"^\s*$", None, regex=True), "comentarios.like_count_text")
atipicos_iqr(comentarios.text.str.len(), "longitud del texto del comentario")

videos.view_count
    n=293  min=2  mediana=1175  media=60430.1  max=8190449
    Q1=215  Q3=7465  IQR=7250  limite_sup=18340
    atipicos: 49  (16.7%)
    cinco mayores: [8190449, 3152619, 749356, 504374, 424874]

comentarios.reply_count
    n=406  min=0  mediana=0  media=0.1  max=7
    Q1=0  Q3=0  IQR=0  limite_sup=0
    atipicos: 30  (7.4%)
    cinco mayores: [7, 5, 3, 3, 3]

comentarios.like_count_text
    n=217  min=1  mediana=2  media=10.7  max=405
    Q1=1  Q3=4  IQR=3  limite_sup=8
    atipicos: 34  (15.7%)
    cinco mayores: [405.0, 329.0, 211.0, 191.0, 113.0]

longitud del texto del comentario
    n=406  min=1  mediana=96  media=139.2  max=1525
    Q1=50  Q3=178  IQR=128  limite_sup=371
    atipicos: 21  (5.2%)
    cinco mayores: [1525, 1237, 1022, 902, 846]



In [11]:
def consistencia(df, id_col, nombre_col, handle_col):
    """Un identificador debe apuntar a un solo nombre y handle, y viceversa."""
    print(f"{id_col} vs {nombre_col} vs {handle_col}")
    print(f"    {id_col} distintos            : {df[id_col].nunique()}")
    print(f"    {nombre_col} distintos        : {df[nombre_col].nunique()}")
    print(f"    un id con varios nombres      : {int((df.groupby(id_col)[nombre_col].nunique() > 1).sum())}")
    print(f"    un nombre con varios id       : {int((df.groupby(nombre_col)[id_col].nunique() > 1).sum())}")
    print(f"    un id con varios handles      : {int((df.groupby(id_col)[handle_col].nunique() > 1).sum())}")
    print(f"    un handle con varios id       : {int((df.groupby(handle_col)[id_col].nunique() > 1).sum())}")
    print()


consistencia(videos, "channel_id", "channel_name", "channel_handle")
consistencia(comentarios, "author_channel_id", "author_name", "author_handle")

mapa_canal = videos.set_index("channel_id").channel_name.to_dict()
desajuste = int((comentarios.channel_id.map(mapa_canal).fillna("") != comentarios.channel_name).sum())
print(f"comentarios cuyo channel_name no coincide con el del catalogo: {desajuste}")

channel_id vs channel_name vs channel_handle
    channel_id distintos            : 97
    channel_name distintos        : 97
    un id con varios nombres      : 0
    un nombre con varios id       : 0
    un id con varios handles      : 0
    un handle con varios id       : 0

author_channel_id vs author_name vs author_handle
    author_channel_id distintos            : 332
    author_name distintos        : 332
    un id con varios nombres      : 0
    un nombre con varios id       : 0
    un id con varios handles      : 0
    un handle con varios id       : 0

comentarios cuyo channel_name no coincide con el del catalogo: 0


In [12]:
print("FORMATO DE LOS IDENTIFICADORES")
for nombre, df, col in [("videos", videos, "video_id"), ("videos", videos, "channel_id"),
                        ("comentarios", comentarios, "comment_id"),
                        ("comentarios", comentarios, "author_channel_id")]:
    largos = df[col].str.len().value_counts().sort_index().to_dict()
    con_espacios = int((df[col] != df[col].str.strip()).sum())
    print(f"  {nombre}.{col:20s} largos={largos}  espacios_extremos={con_espacios}")

print()
print("FORMATO DE LOS HANDLES")
print(f"  channel_handle empieza con '/@'          : {videos.channel_handle.str.startswith('/@').mean():.1%}")
print(f"  author_handle empieza con '/@'           : {comentarios.author_handle.str.startswith('/@').mean():.1%}")
print(f"  author_handle == '/' + author_name       : {(comentarios.author_handle == '/' + comentarios.author_name).mean():.1%}")
print(f"  author_handle con codificacion porcentual: {int(comentarios.author_handle.str.contains('%').sum())}")

FORMATO DE LOS IDENTIFICADORES
  videos.video_id             largos={11: 293}  espacios_extremos=0
  videos.channel_id           largos={24: 293}  espacios_extremos=0
  comentarios.comment_id           largos={26: 370, 49: 36}  espacios_extremos=0
  comentarios.author_channel_id    largos={24: 406}  espacios_extremos=0

FORMATO DE LOS HANDLES
  channel_handle empieza con '/@'          : 100.0%
  author_handle empieza con '/@'           : 100.0%
  author_handle == '/' + author_name       : 96.6%
  author_handle con codificacion porcentual: 14


In [13]:
comentarios[comentarios.author_handle.str.contains("%")][["author_name", "author_handle"]].drop_duplicates()

,author_name,author_handle
73,@ErvinLeonardoCarreraLatín,/@ErvinLeonardoCarreraLat%C3%ADn
170,@AlejandroPérez-b6r,/@AlejandroP%C3%A9rez-b6r
230,@ErmePérez-q4s,/@ErmeP%C3%A9rez-q4s
251,@BorisTebalán,/@BorisTebal%C3%A1n
296,@RolandoCastroPérez-d7y,/@RolandoCastroP%C3%A9rez-d7y
309,@OdilioRodríguez-c5w,/@OdilioRodr%C3%ADguez-c5w
322,@IvánPérez-j4j,/@Iv%C3%A1nP%C3%A9rez-j4j
368,@JoséLopez-g6k,/@Jos%C3%A9Lopez-g6k
393,@NormaLópez-v7e,/@NormaL%C3%B3pez-v7e


In [14]:
# comment_id de una respuesta tiene la forma <id_del_padre>.<id_de_la_respuesta>
es_respuesta = comentarios.comment_id.str.contains(r"\.", regex=True)
principales = comentarios[~es_respuesta]
respuestas = comentarios[es_respuesta]

id_padre = respuestas.comment_id.str.split(".").str[0]
autor_por_comentario = comentarios.set_index("comment_id").author_channel_id

print("ESTRUCTURA DE comment_id")
print(f"  comentarios principales (largo 26)      : {len(principales)}")
print(f"  respuestas (largo 49, con punto)        : {len(respuestas)}")
print(f"  respuestas cuyo padre esta en el archivo: {int(id_padre.isin(comentarios.comment_id).sum())} de {len(respuestas)}")
print()
print(f"  respuestas declaradas por reply_count   : {principales.reply_count.astype(int).sum()}")
print(f"  respuestas presentes como fila          : {len(respuestas)}")
print(f"  videos en los que aparecen respuestas   : {respuestas.video_id.nunique()}")

ESTRUCTURA DE comment_id
  comentarios principales (largo 26)      : 370
  respuestas (largo 49, con punto)        : 36
  respuestas cuyo padre esta en el archivo: 36 de 36

  respuestas declaradas por reply_count   : 51
  respuestas presentes como fila          : 36
  videos en los que aparecen respuestas   : 8


- **Dimensiones**: 293 videos con 20 variables y 406 comentarios con 17 variables, 12,762 celdas en
  total. Ninguna columna llega tipada, las 37 se cargan como `str`.
- **Tipos de variable**: predominan las cualitativas. De las 37 variables, 28 son cualitativas
  nominales, 2 son cualitativas ordinales, 5 son cuantitativas discretas y 2 son cuantitativas
  continuas. Cabe mencionar que la mayoría de las nominales son identificadores o texto libre, por lo
  que sirven para construir nodos y no para describir distribuciones.
- **Valores faltantes**: no vienen como nulos sino en tres formas distintas. Cadena vacía en
  `description` (26), `description_snippet` (25), `published_time` (13), `view_count_text` (13) y
  `viewer_rating` (406); solo espacios en `like_count_text` (189); y lista vacía `[]` en `keywords`
  (162, el 55.3 % del catálogo).
- **Duplicados**: ninguna fila completa se repite y ningún identificador se repite en los dos
  archivos, es decir, las llaves primarias son sólidas.
- **Títulos repetidos que no son duplicados**: 19 videos comparten título con otro del mismo canal,
  pero tienen `video_id` distinto. Son conferencias de prensa recurrentes, hasta 6 videos con el
  mismo nombre, de tal forma que eliminarlos por título borraría contenido legítimo.
- **Variables constantes**: `is_pinned` toma `False` en los 406 comentarios y `viewer_rating` está
  vacía en los 406, por lo que ninguna de las dos puede diferenciar nada. En videos no hay ninguna.
- **Valores atípicos**: `view_count` tiene 49 atípicos por la regla de Tukey (16.7 %), con un máximo
  de 8,190,449 frente a una mediana de 1,175, es decir, la distribución está muy sesgada a la
  derecha. En `reply_count` el rango intercuartílico es 0, por lo que cualquier valor mayor a cero
  cuenta como atípico, y eso más que anomalía refleja que casi ningún comentario recibe respuestas.
- **Consistencia entre identificadores, nombres y handles**: es perfecta. Ningún `channel_id` apunta
  a más de un nombre o handle ni al contrario, y lo mismo ocurre con `author_channel_id`. Los 97
  canales tienen 97 nombres distintos y los 332 autores 332 nombres distintos.
- **Handles mal codificados**: 14 `author_handle` traen codificación porcentual, por ejemplo
  `/@AlejandroP%C3%A9rez-b6r` en lugar de `@AlejandroPérez-b6r`, lo cual baja la coincidencia con
  `author_name` de un 100 % a un 96.6 %.
- **Comentarios que no son principales**: `comment_id` presenta dos longitudes, 26 y 49. Los de 49
  llevan un punto y tienen la forma `id_del_padre.id_de_la_respuesta`, de tal forma que 36 de los 406
  registros son respuestas y no comentarios de primer nivel, aunque el enunciado describa el archivo
  como si todos lo fueran.

---

## **2.2. Variables que no pueden utilizarse o que requieren precaución**

| Archivo | Variable | Problema | Tratamiento y razón |
|---|---|---|---|
| comentarios | `viewer_rating` | Vacía en los 406 registros | Se descarta. No hay ningún valor que analizar, por lo que conservarla solo agregaría una columna muerta al conjunto. |
| comentarios | `is_pinned` | Constante en `False` | Se descarta. Una variable sin variabilidad no puede diferenciar nodos ni explicar nada, ya que todos los registros caen en la misma categoría. |
| videos | `upload_date` | Idéntica a `publish_date` en el 100 % | Se descarta y se conserva `publish_date`. Mantener las dos duplicaría la misma información temporal sin aportar nada. |
| videos | `owner_handle` | Idéntica a `channel_handle` en el 100 % | Se descarta y se conserva `channel_handle`, por la misma razón de redundancia exacta. |
| comentarios | `video_title` | Repite `videos.title` en el 100 % | Se descarta tras la integración, ya que el título se obtiene uniendo por `video_id` y no hace falta arrastrarlo en el archivo de comentarios. |
| comentarios | `channel_name` | Repite `videos.channel_name` en el 100 % | Se descarta tras la integración. Adicional, su nombre confunde, porque es el canal dueño del video y no el autor del comentario. |
| videos | `published_time` | Tiempo relativo del tipo "hace 2 días" | Se conserva pero no se convierte a fecha. Depende del momento exacto de la recolección, de tal forma que volverla fecha absoluta sería inventar el dato. |
| comentarios | `published_text` | Tiempo relativo del tipo "hace 6 meses" | Se conserva sin convertir, por la misma razón. Sirve para ordenar de forma aproximada, no para ubicar en el calendario. |
| videos | `view_count_text` | Conteo en texto que difiere de `view_count` en 53 de 280 casos | Se convierte a numérico pero se usa `view_count` para los análisis. Las dos medidas se capturaron en momentos distintos, así que se prefiere la que ya viene como entero. |
| comentarios | `like_count_text` | 189 registros vienen como un espacio en blanco | Se convierte a numérico y el blanco se toma como cero. YouTube no muestra número cuando un comentario no tiene "me gusta", por lo que el blanco es su forma de representar el cero. |
| comentarios | `author_name` | Empieza con `@` en el 100 % de los registros | Se usa solo como etiqueta visible. En realidad es un handle y los handles se pueden cambiar, es por esto que los nodos de autor se construyen con `author_channel_id`. |
| comentarios | `channel_id` | Es el canal dueño del video, no el del autor | Se conserva pero se documenta el riesgo. Usarla como identificador de autor rompería la red por completo, ya que confundiría al emisor con el receptor. |
| comentarios | `reply_count` | Cuenta respuestas sin identificar quién las escribió | No se usa como arista. Declara 51 respuestas de las cuales solo 36 están presentes como fila, por lo que la conversación no se puede reconstruir por completo. |
| comentarios | `comment_id` | Mezcla comentarios principales y respuestas | Se conserva como llave primaria pero se deriva una marca de tipo, ya que 36 registros son respuestas y tratarlos como comentarios de primer nivel sesgaría cualquier conteo. |
| videos | `source_query` | Describe el muestreo, no el tema del video | Se usa solo para discutir el sesgo de recolección. Un video pudo entrar por la consulta `guatemala lluvias` sin que ese sea su tema central. |
| videos | `keywords` | Lista en texto, vacía en 162 de 293 registros | Se convierte a lista real pero se usa con reserva, ya que más de la mitad del catálogo no tiene etiquetas y cualquier conteo estaría sesgado hacia los videos que sí las traen. |
| videos | `description` | Vacía en 26 registros | Se conserva y el vacío se deja como faltante honesto, dado que la descripción simplemente no se capturó y rellenarla no tendría sustento. |

---

## **2.3. Normalización de identificadores y nombres**

In [15]:
from urllib.parse import unquote

videos_norm = videos.copy()
comentarios_norm = comentarios.copy()

COLUMNAS_ID = {
    "videos": ["video_id", "channel_id"],
    "comentarios": ["comment_id", "video_id", "channel_id", "author_channel_id"],
}

cambios_id = 0
for col in COLUMNAS_ID["videos"]:
    cambios_id += int((videos_norm[col] != videos_norm[col].str.strip()).sum())
    videos_norm[col] = videos_norm[col].str.strip()
for col in COLUMNAS_ID["comentarios"]:
    cambios_id += int((comentarios_norm[col] != comentarios_norm[col].str.strip()).sum())
    comentarios_norm[col] = comentarios_norm[col].str.strip()

print(f"[ids]  valores con espacio sobrante corregidos: {cambios_id}")

[ids]  valores con espacio sobrante corregidos: 0


In [16]:
def normaliza_handle(serie):
    """Quita la barra inicial y decodifica la codificacion porcentual de las tildes."""
    return serie.str.strip().str.lstrip("/").map(unquote)


videos_norm["handle_canal"] = normaliza_handle(videos_norm.channel_handle)
comentarios_norm["handle_autor"] = normaliza_handle(comentarios_norm.author_handle)

for col in ["title", "channel_name"]:
    videos_norm[col] = videos_norm[col].str.strip()
for col in ["video_title", "channel_name", "author_name"]:
    comentarios_norm[col] = comentarios_norm[col].str.strip()

print("HANDLES NORMALIZADOS")
print(f"  handles de canal modificados : {int((videos_norm.handle_canal != videos_norm.channel_handle).sum())}")
print(f"  handles de autor modificados : {int((comentarios_norm.handle_autor != comentarios_norm.author_handle).sum())}")
print(f"  handles con '%' restantes    : {int(comentarios_norm.handle_autor.str.contains('%').sum())}")
print(f"  handle_autor coincide con author_name: {(comentarios_norm.handle_autor == comentarios_norm.author_name).mean():.1%}")
print()
comentarios_norm[comentarios_norm.author_handle.str.contains("%")][["author_name", "author_handle", "handle_autor"]].drop_duplicates()

HANDLES NORMALIZADOS
  handles de canal modificados : 293
  handles de autor modificados : 406
  handles con '%' restantes    : 0
  handle_autor coincide con author_name: 100.0%



,author_name,author_handle,handle_autor
73,@ErvinLeonardoCarreraLatín,/@ErvinLeonardoCarreraLat%C3%ADn,@ErvinLeonardoCarreraLatín
170,@AlejandroPérez-b6r,/@AlejandroP%C3%A9rez-b6r,@AlejandroPérez-b6r
230,@ErmePérez-q4s,/@ErmeP%C3%A9rez-q4s,@ErmePérez-q4s
251,@BorisTebalán,/@BorisTebal%C3%A1n,@BorisTebalán
296,@RolandoCastroPérez-d7y,/@RolandoCastroP%C3%A9rez-d7y,@RolandoCastroPérez-d7y
309,@OdilioRodríguez-c5w,/@OdilioRodr%C3%ADguez-c5w,@OdilioRodríguez-c5w
322,@IvánPérez-j4j,/@Iv%C3%A1nP%C3%A9rez-j4j,@IvánPérez-j4j
368,@JoséLopez-g6k,/@Jos%C3%A9Lopez-g6k,@JoséLopez-g6k
393,@NormaLópez-v7e,/@NormaL%C3%B3pez-v7e,@NormaLópez-v7e


In [17]:
print("CONTROL: LOS IDENTIFICADORES NO FUERON SUSTITUIDOS POR NOMBRES")
print(f"  videos.video_id intacto           : {videos_norm.video_id.equals(videos.video_id)}")
print(f"  videos.channel_id intacto         : {videos_norm.channel_id.equals(videos.channel_id)}")
print(f"  comentarios.comment_id intacto    : {comentarios_norm.comment_id.equals(comentarios.comment_id)}")
print(f"  comentarios.author_channel_id ok  : {comentarios_norm.author_channel_id.equals(comentarios.author_channel_id)}")
print()
print(f"  canales identificados por channel_id        : {videos_norm.channel_id.nunique()}")
print(f"  autores identificados por author_channel_id : {comentarios_norm.author_channel_id.nunique()}")

CONTROL: LOS IDENTIFICADORES NO FUERON SUSTITUIDOS POR NOMBRES
  videos.video_id intacto           : True
  videos.channel_id intacto         : True
  comentarios.comment_id intacto    : True
  comentarios.author_channel_id ok  : True

  canales identificados por channel_id        : 97
  autores identificados por author_channel_id : 332


---

## **2.4. Conversión a numérico de las variables de conteo**

In [18]:
print("FORMATOS PRESENTES EN LAS VARIABLES DE CONTEO")
patron = videos.view_count_text.str.replace(r"[\d.,]", "#", regex=True).value_counts()
print("view_count_text (digitos sustituidos por #):")
print(patron.to_string())
print()
print(f"  abreviaturas K, M, mil o millones : {int(videos.view_count_text.str.contains(r'K|M|mil|millones', case=False, regex=True).sum())}")
print(f"  separador de miles observado      : coma")
print()
print("like_count_text: valores mas frecuentes")
print(comentarios.like_count_text.value_counts().head(6).to_string())
print()
print("reply_count: valores presentes")
print(comentarios.reply_count.value_counts().sort_index().to_string())

FORMATOS PRESENTES EN LAS VARIABLES DE CONTEO
view_count_text (digitos sustituidos por #):
view_count_text
##### vistas        86
### vistas          81
###### vistas       44
## vistas           39
####### vistas      17
                    13
# vistas            11
######### vistas     2

  abreviaturas K, M, mil o millones : 0
  separador de miles observado      : coma

like_count_text: valores mas frecuentes
like_count_text
     189
1     88
2     38
3     26
4     11
5      6

reply_count: valores presentes
reply_count
0    376
1     21
2      3
3      4
5      1
7      1


In [19]:
def a_numero(serie):
    """Extrae el numero quitando separadores de miles y texto acompañante."""
    limpio = serie.str.strip().str.replace(",", "", regex=False).str.extract(r"(\d+)")[0]
    return pd.to_numeric(limpio, errors="coerce")


videos_norm["vistas"] = a_numero(videos_norm.view_count)
videos_norm["vistas_texto"] = a_numero(videos_norm.view_count_text)
comentarios_norm["respuestas"] = a_numero(comentarios_norm.reply_count).astype(int)

en_blanco = comentarios_norm.like_count_text.str.strip() == ""
me_gusta_bruto = a_numero(comentarios_norm.like_count_text)
solo_de_blancos = bool((me_gusta_bruto.isna() == en_blanco).all())
comentarios_norm["me_gusta"] = me_gusta_bruto.fillna(0).astype(int)

print("TRATAMIENTO DE like_count_text")
print(f"  valores en blanco                      : {int(en_blanco.sum())}")
print(f"  los NaN provienen solo de esos blancos : {solo_de_blancos}")
print(f"  comentarios con me_gusta > 0           : {int((comentarios_norm.me_gusta > 0).sum())}")
print()

print("RESULTADO DE LA CONVERSION")
for nombre, serie, origen in [
    ("vistas", videos_norm.vistas, videos_norm.view_count),
    ("vistas_texto", videos_norm.vistas_texto, videos_norm.view_count_text),
    ("me_gusta", comentarios_norm.me_gusta.astype(float), comentarios_norm.like_count_text),
    ("respuestas", comentarios_norm.respuestas.astype(float), comentarios_norm.reply_count),
]:
    print(f"  {nombre:14s} convertidos={serie.notna().sum():4d}  NaN={int(serie.isna().sum()):4d}  "
          f"origen_vacio={int((origen.str.strip() == '').sum()):4d}")

TRATAMIENTO DE like_count_text
  valores en blanco                      : 189
  los NaN provienen solo de esos blancos : True
  comentarios con me_gusta > 0           : 217

RESULTADO DE LA CONVERSION
  vistas         convertidos= 293  NaN=   0  origen_vacio=   0
  vistas_texto   convertidos= 280  NaN=  13  origen_vacio=  13
  me_gusta       convertidos= 406  NaN=   0  origen_vacio= 189
  respuestas     convertidos= 406  NaN=   0  origen_vacio=   0


In [20]:
comparables = videos_norm[["vistas", "vistas_texto"]].dropna()
diferentes = comparables[comparables.vistas != comparables.vistas_texto]

print("COHERENCIA ENTRE LAS DOS MEDIDAS DE VISUALIZACIONES")
print(f"  registros comparables       : {len(comparables)}")
print(f"  con el mismo valor          : {len(comparables) - len(diferentes)}")
print(f"  con valor distinto          : {len(diferentes)}  ({len(diferentes) / len(comparables):.1%})")
print(f"  diferencia absoluta mediana : {(diferentes.vistas - diferentes.vistas_texto).abs().median():.0f}")
print(f"  diferencia absoluta maxima  : {(diferentes.vistas - diferentes.vistas_texto).abs().max():.0f}")
print()
diferentes.head(8)

COHERENCIA ENTRE LAS DOS MEDIDAS DE VISUALIZACIONES
  registros comparables       : 280
  con el mismo valor          : 227
  con valor distinto          : 53  (18.9%)
  diferencia absoluta mediana : 4
  diferencia absoluta maxima  : 3382



,vistas,vistas_texto
0,2357,2390.0
5,6099,6098.0
6,15455,15436.0
16,5181,5180.0
25,4484,4485.0
27,27111,27107.0
33,741,745.0
49,5,3.0


In [21]:
print("TIPOS RESULTANTES")
print(videos_norm[["vistas", "vistas_texto"]].dtypes.to_string())
print(comentarios_norm[["me_gusta", "respuestas"]].dtypes.to_string())
print()
print(videos_norm[["vistas"]].describe().round(1).to_string())
print()
print(comentarios_norm[["me_gusta", "respuestas"]].describe().round(2).to_string())

TIPOS RESULTANTES
vistas            int64
vistas_texto    float64
me_gusta      int64
respuestas    int64

          vistas
count      293.0
mean     60430.1
std     515798.3
min          2.0
25%        215.0
50%       1175.0
75%       7465.0
max    8190449.0

       me_gusta  respuestas
count    406.00      406.00
mean       5.73        0.13
std       30.66        0.58
min        0.00        0.00
25%        0.00        0.00
50%        1.00        0.00
75%        2.00        0.00
max      405.00        7.00


- **Separadores**: el único separador presente es la coma de miles en `view_count_text`, que se
  elimina antes de convertir, y no se observa ningún separador decimal porque los cuatro conteos son
  enteros.
- **Abreviaturas**: no se encontró ninguna abreviatura del tipo K, M, mil o millones, de tal forma
  que no hizo falta ninguna regla de expansión.
- **Texto acompañante**: `view_count_text` trae siempre la palabra "vistas" pegada al número, que se
  descarta extrayendo únicamente los dígitos.
- **Valores no válidos**: los 13 `view_count_text` vacíos quedan como faltantes, ya que el dato no se
  capturó, mientras que los 189 `like_count_text` en blanco se convierten a cero porque YouTube omite
  el número cuando un comentario no tiene "me gusta"; se verificó que esos blancos son el único
  origen de valores no convertibles en esa columna.

---

## **2.5. Creación de texto_original y texto_limpio**

In [22]:
comentarios_norm["texto_original"] = comentarios_norm.text

print("TEXTO ORIGINAL")
print(f"  registros                   : {len(comentarios_norm)}")
print(f"  identico a la columna text  : {comentarios_norm.texto_original.equals(comentarios.text)}")
print(f"  longitud media (caracteres) : {comentarios_norm.texto_original.str.len().mean():.1f}")
print(f"  vacios o solo espacios      : {int((comentarios_norm.texto_original.str.strip() == '').sum())}")

TEXTO ORIGINAL
  registros                   : 406
  identico a la columna text  : True
  longitud media (caracteres) : 139.2
  vacios o solo espacios      : 0


---

## **2.6. Decisiones de limpieza aplicadas a texto_limpio**

In [23]:
import re

import emoji as libreria_emoji
import spacy

nlp = spacy.load("es_core_news_sm", disable=["ner", "parser"])
STOPWORDS = nlp.Defaults.stop_words

RE_INVISIBLES = re.compile(r"[​‌‍﻿\xa0]")
RE_URL = re.compile(r"https?://\S+|www\.\S+")
RE_HASHTAG = re.compile(r"#\w+")
RE_MENCION = re.compile(r"@[\w\-.]+")
RE_ESPACIOS = re.compile(r"\s+")

print("Modelo cargado:", nlp.meta["lang"], nlp.meta["name"], nlp.meta["version"])
print("Stopwords en español disponibles:", len(STOPWORDS))

Modelo cargado: es core_news_sm 3.8.0
Stopwords en español disponibles: 521


In [24]:
def quita_invisibles(t):
    return RE_ESPACIOS.sub(" ", RE_INVISIBLES.sub(" ", t)).strip()


texto = comentarios_norm.texto_original.map(quita_invisibles)

urls = texto.map(RE_URL.findall)
hashtags = texto.map(RE_HASHTAG.findall)
menciones = texto.map(RE_MENCION.findall)
emojis = texto.map(lambda t: [c["emoji"] for c in libreria_emoji.emoji_list(t)])

comentarios_norm["urls"] = urls
comentarios_norm["hashtags"] = hashtags
comentarios_norm["menciones"] = menciones
comentarios_norm["emojis"] = emojis

print("ELEMENTOS INVENTARIADOS EN SU PROPIA COLUMNA")
for nombre, serie in [("urls", urls), ("hashtags", hashtags), ("menciones", menciones), ("emojis", emojis)]:
    con_elemento = int((serie.str.len() > 0).sum())
    print(f"  {nombre:10s} comentarios que lo contienen: {con_elemento:4d} ({con_elemento / len(serie):5.1%})   total: {int(serie.str.len().sum())}")

ELEMENTOS INVENTARIADOS EN SU PROPIA COLUMNA
  urls       comentarios que lo contienen:    1 ( 0.2%)   total: 1
  hashtags   comentarios que lo contienen:    1 ( 0.2%)   total: 1
  menciones  comentarios que lo contienen:    5 ( 1.2%)   total: 5
  emojis     comentarios que lo contienen:   61 (15.0%)   total: 199


In [25]:
from collections import Counter

# Solo se corrigen formas que no existen en español o que son una version sin
# tilde de otra palabra ya presente en el corpus. No se despojan tildes: la
# forma correcta es la que se conserva, ya que en español la tilde cambia el
# significado.
CORRECCIONES_TILDE = {
    "africa": "áfrica", "arevalo": "arévalo", "arzu": "arzú", "baldizon": "baldizón",
    "carcel": "cárcel", "catedratico": "catedrático", "corazon": "corazón",
    "corrupcion": "corrupción", "desnutricion": "desnutrición", "educacion": "educación",
    "excelentisimo": "excelentísimo", "galon": "galón", "informacion": "información",
    "investigacion": "investigación", "jamas": "jamás", "ladron": "ladrón",
    "mexico": "méxico", "minimo": "mínimo", "obligacion": "obligación", "ojala": "ojalá",
    "oposicion": "oposición", "pais": "país", "poblacion": "población", "policia": "policía",
    "publico": "público", "rapido": "rápido", "razon": "razón", "rodriguez": "rodríguez",
    "salvadoreno": "salvadoreño", "senor": "señor", "sen̈or": "señor",
    "sinverguenza": "sinvergüenza", "tecnologia": "tecnología", "unico": "único",
    "verguenza": "vergüenza", "vídeo": "video",
}

CORRECCIONES_ESCRITURA = {
    "exelente": "excelente", "jente": "gente", "cuidad": "ciudad", "ambre": "hambre",
    "president": "presidente", "corruption": "corrupción", "haci": "así",
}

# Pares que difieren solo en la tilde pero que NO se fusionan, porque son
# palabras distintas y el contexto confirma que cada forma es la correcta.
NO_FUSIONADAS = [
    ("ano", "año", "ano y año son palabras distintas"),
    ("renuncie", "renuncié", "el comentario usa el imperativo, no el pasado"),
    ("echo", "hecho", "echo es forma valida del verbo echar"),
]

CORRECCIONES = {**CORRECCIONES_TILDE, **CORRECCIONES_ESCRITURA}
CORRECCIONES_APLICADAS = Counter()

print(f"correcciones de tilde     : {len(CORRECCIONES_TILDE)}")
print(f"correcciones de escritura : {len(CORRECCIONES_ESCRITURA)}")
print(f"pares excluidos a proposito: {len(NO_FUSIONADAS)}")
for origen, destino, motivo in NO_FUSIONADAS:
    print(f"    {origen} no se fusiona con {destino}: {motivo}")

correcciones de tilde     : 36
correcciones de escritura : 7
pares excluidos a proposito: 3
    ano no se fusiona con año: ano y año son palabras distintas
    renuncie no se fusiona con renuncié: el comentario usa el imperativo, no el pasado
    echo no se fusiona con hecho: echo es forma valida del verbo echar


In [26]:
sin_url = texto.map(lambda t: RE_URL.sub(" ", t))
sin_hashtag = sin_url.map(lambda t: RE_HASHTAG.sub(" ", t))
sin_mencion = sin_hashtag.map(lambda t: RE_ESPACIOS.sub(" ", RE_MENCION.sub(" ", t)).strip())


def segmenta(t):
    """Parte el texto en tramos alternos de palabras y de emojis, conservando el orden."""
    piezas, pos = [], 0
    for encontrado in libreria_emoji.emoji_list(t):
        piezas.append(("texto", t[pos:encontrado["match_start"]]))
        piezas.append(("emoji", encontrado["emoji"]))
        pos = encontrado["match_end"]
    piezas.append(("texto", t[pos:]))
    return piezas


def lematiza_tramos(tramos):
    salida = []
    for doc in nlp.pipe(tramos, batch_size=64):
        lemas = []
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space or token.like_num:
                continue
            # Un verbo con pronombre enclitico da un lema de varias palabras,
            # por ejemplo "verbose" produce "verbo el", asi que se filtra cada parte.
            for parte in token.lemma_.lower().split():
                if parte in CORRECCIONES:
                    CORRECCIONES_APLICADAS[(parte, CORRECCIONES[parte])] += 1
                    parte = CORRECCIONES[parte]
                if parte not in STOPWORDS and len(parte) > 1:
                    lemas.append(parte)
        salida.append(" ".join(lemas))
    return salida


def limpia_conservando_emojis(textos):
    plan = [segmenta(t) for t in textos]
    lemas = iter(lematiza_tramos([c for seg in plan for tipo, c in seg if tipo == "texto"]))
    salida = []
    for seg in plan:
        partes = [next(lemas) if tipo == "texto" else contenido for tipo, contenido in seg]
        salida.append(RE_ESPACIOS.sub(" ", " ".join(p for p in partes if p)).strip())
    return salida


comentarios_norm["texto_limpio"] = limpia_conservando_emojis(sin_mencion.tolist())

print("Ejemplo de la transformacion")
for i in [0, 3, 38, 187]:
    print(f"\n  original : {comentarios_norm.texto_original.iloc[i][:110]}")
    print(f"  limpio   : {comentarios_norm.texto_limpio.iloc[i][:110]}")

Ejemplo de la transformacion

  original : Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel
  limpio   : corrupto amigo viejo fiscal verbo cárcel

  original : Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la sombra.
  limpio   : mafioso mazariegos cárcel tiempo sombra

  original : 😮
  limpio   : 😮

  original : 👏👏👏👏👏👏
  limpio   : 👏 👏 👏 👏 👏 👏


In [27]:
print(f"CORRECCIONES ORTOGRAFICAS APLICADAS: {sum(CORRECCIONES_APLICADAS.values())} en {len(CORRECCIONES_APLICADAS)} formas distintas")
print()
for (origen, destino), n in CORRECCIONES_APLICADAS.most_common():
    print(f"  {n:3d}x  {origen:16s} -> {destino}")

sin_usar = sorted(set(CORRECCIONES) - {o for o, _ in CORRECCIONES_APLICADAS})
print()
print(f"entradas del mapa que no aparecieron en el corpus: {len(sin_usar)}  {sin_usar}")

CORRECCIONES ORTOGRAFICAS APLICADAS: 107 en 43 formas distintas

   21x  pais             -> país
   11x  arevalo          -> arévalo
    7x  corrupcion       -> corrupción
    5x  mexico           -> méxico
    4x  vídeo            -> video
    4x  sinverguenza     -> sinvergüenza
    3x  carcel           -> cárcel
    3x  president        -> presidente
    3x  unico            -> único
    3x  ojala            -> ojalá
    3x  cuidad           -> ciudad
    2x  haci             -> así
    2x  corruption       -> corrupción
    2x  exelente         -> excelente
    2x  jente            -> gente
    2x  ambre            -> hambre
    2x  rodriguez        -> rodríguez
    2x  arzu             -> arzú
    2x  obligacion       -> obligación
    1x  investigacion    -> investigación
    1x  salvadoreno      -> salvadoreño
    1x  publico          -> público
    1x  oposicion        -> oposición
    1x  catedratico      -> catedrático
    1x  educacion        -> educación
    1x  poblacion 

In [28]:
ETAPAS = [
    ("0. texto original", comentarios_norm.texto_original),
    ("1. sin caracteres invisibles", texto),
    ("2. sin URL", sin_url),
    ("3. sin hashtags", sin_hashtag),
    ("4. sin menciones", sin_mencion),
    ("5. lematizado, emojis conservados", comentarios_norm.texto_limpio),
]

filas, anterior = [], None
for nombre, serie in ETAPAS:
    filas.append({
        "etapa": nombre,
        "modificados_vs_anterior": 0 if anterior is None else int((serie != anterior).sum()),
        "long_media": round(serie.str.len().mean(), 1),
        "tokens_totales": int(serie.str.split().str.len().sum()),
        "vocabulario": len(set(" ".join(serie).lower().split())),
        "vacios": int((serie.str.strip() == "").sum()),
    })
    anterior = serie

pd.DataFrame(filas)

,etapa,modificados_vs_anterior,long_media,tokens_totales,vocabulario,vacios
0,0. texto original,0,139.2,9769,3171,0
1,1. sin caracteres invisibles,109,138.6,9764,3170,0
2,2. sin URL,1,138.4,9763,3169,0
3,3. sin hashtags,1,138.4,9762,3168,0
4,4. sin menciones,7,138.2,9757,3164,0
5,"5. lematizado, emojis conservados",402,76.0,4143,2009,3


| Elemento | Tratamiento | Justificación |
|---|---|---|
| Caracteres invisibles | Eliminados | Espacios de ancho cero y saltos de línea que rompen la tokenización y pegan palabras que deberían ir separadas. |
| URL | Separadas a la columna `urls` | No aportan contenido semántico al texto, pero se conservan aparte por si interesa analizar a dónde enlazan. |
| Hashtags | Separados a la columna `hashtags` | El inciso pide separarlos, no borrarlos, ya que identifican campañas y temas propios de la conversación. |
| Menciones | Separadas a la columna `menciones` | Indican a quién se dirige el comentario, de tal forma que guardarlas permite recuperar esa señal sin ensuciar el texto lematizado. |
| Emojis | Conservados en `texto_limpio` e inventariados en `emojis` | En comentarios de YouTube el emoji carga la carga afectiva del mensaje, es por esto que quitarlo dejaría vacíos varios comentarios que sí comunican algo. |
| Mayúsculas | Convertidas a minúscula sobre el lema | Se aplica después de lematizar, porque el lematizador rinde peor cuando pierde las mayúsculas originales de nombres propios. |
| Puntuación | Eliminada por el atributo `is_punct` de spaCy | Se usa el tokenizador en lugar de una expresión regular para no partir palabras con guion ni apóstrofo. |
| Números | Eliminados por el atributo `like_num` | Cubre tanto dígitos como números escritos con letra, que una expresión regular dejaría pasar. |
| Stopwords | Eliminadas con la lista en español de spaCy | Artículos, preposiciones y pronombres dominan la frecuencia sin aportar tema alguno. |
| Lematización | Aplicada con `es_core_news_sm` | Reduce las flexiones a su forma base, por ejemplo `diputados` a `diputado` y `protestaba` a `protestar`, de tal forma que las variantes no se cuenten por separado. |
| Verbos con pronombre enclítico | Lema separado y filtrado parte por parte | spaCy devuelve un lema de varias palabras, por ejemplo `verbose` produce `verbo el`, y el pronombre debe caer aunque el token completo no se marque como stopword. |
| Palabras mal escritas | Corregidas con un mapa curado de 43 entradas | Se probó un corrector automático y dañaba palabras válidas, por ejemplo convertía `mafioso` en `marioso` y `arevalo` en `regalo`, de tal forma que se prefirió un mapa revisado a mano y auditable. |
| Tildes faltantes | Unificadas hacia la forma acentuada correcta | No se despojan tildes, ya que en español cambian el significado; lo que se hace es lo contrario, llevar `pais` a `país` solo cuando la forma correcta ya existe en el corpus. |
| Pares ambiguos por tilde | Deliberadamente no fusionados | `ano` y `año`, `renuncie` y `renuncié`, `echo` y `hecho` son palabras distintas, y el contexto de los comentarios confirma que cada forma está bien escrita. |
| Tokens de un carácter | Eliminados | Son restos de la limpieza que no llegan a formar palabra. |

---

## **2.7. Efecto cuantificado de la limpieza**

In [29]:
antes = comentarios_norm.texto_original
despues = comentarios_norm.texto_limpio

print("REGISTROS")
print(f"  registros al inicio               : {len(comentarios)}")
print(f"  registros al final                : {len(comentarios_norm)}")
print(f"  registros eliminados              : {len(comentarios) - len(comentarios_norm)}")
print(f"  registros con el texto modificado : {int((antes != despues).sum())}  ({(antes != despues).mean():.1%})")
print()

print("TEXTOS VACIOS")
print(f"  vacios antes                      : {int((antes.str.strip() == '').sum())}")
print(f"  vacios despues                    : {int((despues.str.strip() == '').sum())}")
print()

print("DUPLICADOS DE TEXTO")
print(f"  duplicados antes                  : {int(antes.duplicated().sum())}")
print(f"  duplicados despues                : {int(despues.duplicated().sum())}")
print(f"  textos distintos antes            : {antes.nunique()}")
print(f"  textos distintos despues          : {despues.nunique()}")

REGISTROS
  registros al inicio               : 406
  registros al final                : 406
  registros eliminados              : 0
  registros con el texto modificado : 402  (99.0%)

TEXTOS VACIOS
  vacios antes                      : 0
  vacios despues                    : 3

DUPLICADOS DE TEXTO
  duplicados antes                  : 2
  duplicados despues                : 6
  textos distintos antes            : 404
  textos distintos despues          : 400


In [30]:
tokens_antes = antes.str.split().str.len()
tokens_despues = despues.str.split().str.len()

vocab_antes = set(" ".join(antes).lower().split())
vocab_despues = set(" ".join(despues).split())

print("LONGITUD Y VOCABULARIO")
print(f"  caracteres, media antes / despues   : {antes.str.len().mean():.1f} / {despues.str.len().mean():.1f}")
print(f"  caracteres, mediana antes / despues : {antes.str.len().median():.0f} / {despues.str.len().median():.0f}")
print(f"  tokens totales antes / despues      : {int(tokens_antes.sum())} / {int(tokens_despues.sum())}")
print(f"  tokens por comentario, media        : {tokens_antes.mean():.1f} / {tokens_despues.mean():.1f}")
print(f"  vocabulario antes / despues         : {len(vocab_antes)} / {len(vocab_despues)}")
print(f"  reduccion de tokens                 : {1 - tokens_despues.sum() / tokens_antes.sum():.1%}")
print(f"  reduccion de vocabulario            : {1 - len(vocab_despues) / len(vocab_antes):.1%}")

LONGITUD Y VOCABULARIO
  caracteres, media antes / despues   : 139.2 / 76.0
  caracteres, mediana antes / despues : 96 / 53
  tokens totales antes / despues      : 9769 / 4143
  tokens por comentario, media        : 24.1 / 10.2
  vocabulario antes / despues         : 3171 / 2009
  reduccion de tokens                 : 57.6%
  reduccion de vocabulario            : 36.6%


In [31]:
vacios_despues = comentarios_norm[comentarios_norm.texto_limpio.str.strip() == ""]

print(f"COMENTARIOS QUE QUEDARON VACIOS TRAS LA LIMPIEZA: {len(vacios_despues)}")
vacios_despues[["comment_id", "texto_original", "emojis"]]

COMENTARIOS QUE QUEDARON VACIOS TRAS LA LIMPIEZA: 3


,comment_id,texto_original,emojis
97,UgwxjmtBoAozs0ZUAcd4AaABAg.AaAbqoGVGipAaAjNG9oPRN,que?,[]
112,Ugx74OPnWkWaFjpFuA94AaABAg.A7ki-1ED5b2A7nIcG2_svP,Cierto.,[]
272,Ugy__nsWCBggw9EzOUd4AaABAg.A7lbRpGUZoHA7nJxaD37u-,Cierto.,[]


In [32]:
duplicados_nuevos = despues[despues.duplicated(keep=False) & (despues.str.strip() != "")]

print(f"TEXTOS LIMPIOS REPETIDOS: {duplicados_nuevos.nunique()} textos distintos en {len(duplicados_nuevos)} filas")
comentarios_norm.loc[duplicados_nuevos.index, ["comment_id", "texto_original", "texto_limpio"]].sort_values("texto_limpio")

TEXTOS LIMPIOS REPETIDOS: 2 textos distintos en 6 filas


,comment_id,texto_original,texto_limpio
116,Ugx8FxDKPAY-M_JhRgd4AaABAg,Ahora se destapo la CORRUPCION Q EXISTIA Y AHO...,destapo corrupción existia contamino ampliacio...
324,Ugz7WPxTOQEwmZpnx_V4AaABAg,Ahora se destapo la CORRUPCION Q EXISTIA Y AHO...,destapo corrupción existia contamino ampliacio...
79,Ugwm9VajlW3ydbsf-BJ4AaABAg,Excelente.,excelente
83,UgwovhZaCiV0r6XOH_R4AaABAg,excelente,excelente
160,UgxW0ZnTohConSPgOpp4AaABAg,Exelente,excelente
246,UgyIteBIiYe7DjUN3554AaABAg,EXCELENTE,excelente


In [33]:
from collections import Counter

frecuencias = Counter(" ".join(despues).split())

print("VEINTE ELEMENTOS MAS FRECUENTES EN texto_limpio")
for palabra, n in frecuencias.most_common(20):
    print(f"  {palabra:18s} {n}")

VEINTE ELEMENTOS MAS FRECUENTES EN texto_limpio
  país               57
  pueblo             56
  guatemala          50
  diputado           49
  pagar              37
  dinero             34
  presidente         34
  corrupto           29
  😂                  29
  trabajo            28
  excelente          24
  seguir             22
  trabajar           21
  empresa            20
  sueldo             20
  🇬🇹                 19
  proyecto           18
  almuerzo           17
  dios               17
  👏                  17


- **No se eliminó ningún registro.** Los 406 comentarios siguen presentes, acorde a la política de no
  descartar filas durante la limpieza, ya que esa decisión corresponde a cada análisis y no al
  preprocesamiento.
- **La limpieza tocó prácticamente todo el corpus**, con 402 de los 406 textos modificados, un
  99.0 %, lo cual es esperable porque casi cualquier comentario trae al menos una stopword o una
  palabra flexionada.
- **El paso más agresivo fue la lematización con eliminación de stopwords**, que por sí sola modificó
  402 comentarios, muy por encima del resto. Los pasos de URL, hashtags y menciones apenas afectaron
  9 comentarios en conjunto, mientras que la limpieza de caracteres invisibles afectó 109.
- **El corpus se redujo a menos de la mitad en volumen**, pasando de 9,769 a 4,143 tokens, una caída
  del 57.6 %, y de 139.2 a 76.0 caracteres de longitud media por comentario.
- **El vocabulario se redujo un 36.6 %**, de 3,171 a 2,009 formas distintas, que es justamente el
  efecto que se busca con la lematización, ya que las variantes de una misma palabra dejan de
  contarse por separado.
- **La corrección ortográfica unificó 107 apariciones repartidas en 43 formas mal escritas**, siendo
  las más frecuentes `pais` hacia `país` con 21 casos y `arevalo` hacia `arévalo` con 11. Cabe
  mencionar que sin esta corrección `país` y `pais` se habrían contado como dos palabras distintas en
  todo el análisis de frecuencias.
- **Conservar los emojis evitó vaciar comentarios**. Solo quedaron vacíos 3 comentarios, todos
  formados por una sola stopword del tipo "Cierto." o "que?", mientras que los que eran únicamente
  emojis siguen aportando contenido al análisis.
- **Los duplicados de texto pasaron de 2 a 6**, y eso no es un defecto sino consecuencia esperada de
  la normalización: "Excelente.", "excelente" y "EXCELENTE" colapsan en un mismo texto limpio, de tal
  forma que ahora se reconocen como el mismo mensaje.

In [34]:
DIR_PROCESSED = RAIZ / "data" / "processed"
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

comentarios_norm["es_respuesta"] = comentarios_norm.comment_id.str.contains(r"\.", regex=True)
comentarios_norm["comment_id_padre"] = (
    comentarios_norm.comment_id.str.split(".").str[0].where(comentarios_norm.es_respuesta)
)

COLUMNAS_COMENTARIOS = [
    "comment_id", "video_id", "author_channel_id", "channel_id", "author_name", "handle_autor",
    "es_respuesta", "comment_id_padre", "me_gusta", "respuestas", "published_text",
    "texto_original", "texto_limpio", "urls", "hashtags", "menciones", "emojis",
]
COLUMNAS_VIDEOS = [
    "video_id", "channel_id", "channel_name", "handle_canal", "title", "category",
    "source_query", "source_group", "query_hits", "keywords", "description",
    "vistas", "vistas_texto", "publish_date", "published_time",
]

RUTA_COMENTARIOS_LIMPIO = DIR_PROCESSED / "comentarios_limpio.csv"
RUTA_VIDEOS_LIMPIO = DIR_PROCESSED / "videos_limpio.csv"

comentarios_norm[COLUMNAS_COMENTARIOS].to_csv(RUTA_COMENTARIOS_LIMPIO, index=False)
videos_norm[COLUMNAS_VIDEOS].to_csv(RUTA_VIDEOS_LIMPIO, index=False)

print(f"[guardado]  {RUTA_COMENTARIOS_LIMPIO.name:24s} -> {len(comentarios_norm)} filas, {len(COLUMNAS_COMENTARIOS)} columnas")
print(f"[guardado]  {RUTA_VIDEOS_LIMPIO.name:24s} -> {len(videos_norm)} filas, {len(COLUMNAS_VIDEOS)} columnas")

[guardado]  comentarios_limpio.csv   -> 406 filas, 17 columnas
[guardado]  videos_limpio.csv        -> 293 filas, 15 columnas
